In [0]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, current_date, lit
from pyspark.sql.utils import AnalysisException

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.raw")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.gold")

spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.raw.arquivos_tc2")



In [0]:

#Função para criar as tabelas delta na camada bronze
def carregar_bronze(
    csv_path: str,
    catalogo: str,
    schema: str,
    tabela: str,
    sep: str = ",",
    encoding: str = "UTF-8",
    header: bool = True,
    infer_schema: bool = True
):
    #Iniciando 
    print(f"Ingestão Raw Bronze: {tabela}")
  
#leitura do arquivo com try catch pra pegar possíveis erros
    try:

        df = (
            spark.read
            .option("header", header)
            .option("inferSchema", infer_schema)
            .option("sep", sep)
            .option("encoding", encoding)
            .option("mode", "FAILFAST")
            .csv(csv_path)
        )

    except Exception as e:
        raise Exception(f"Erro ao ler CSV:\n{e}")

    #verifica se o arquivo está vazio
    if df.isEmpty():
        raise Exception("Arquivo vazio.")
    
    #aqui vejo se o arquivo tem uma coluna só (problema de delimitador errado)
    if len(df.columns) <= 1:
        raise Exception(
            "Provável delimitador incorreto."
        )

    #verifica se tem colunas com dois nomes. Decidi fazer aqui pq pode dar problema na criação da tabela na bronze em vez de fazer na silver
    if len(df.columns) != len(set(df.columns)):
        raise Exception(
            "Existem colunas duplicadas."
        )

    #Adiciono timestamp, data_ref e o nome do arquivo origem 
    df = (
        df
        .withColumn("dt_ingestao", current_timestamp())
        .withColumn("dt_referencia_ingestao", current_date())
        .withColumn("arquivo_origem", lit(csv_path))
    )

    nome_tabela = f"{catalogo}.{schema}.{tabela}"

    #coloquei o append pra manter o histórico particionado
    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(nome_tabela)
    )

    #printo no console para ver se deu tudo certo
    print(f"Tabela criada: {nome_tabela}")
    print(f"Linhas carregadas: {df.count():,}")

    print("Carga Bronze finalizada.")



In [0]:

#Definindo o caminho dos arquivos no volume do schema RAW
path_meta_brasil = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv"
path_meta_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv"
path_meta_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv"
path_avaliacao_municipio = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_municipio.csv"
path_avaliacao_uf = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_uf.csv"
path_avaliacao_alunos = "/Volumes/workspace/raw/arquivos_tc2/br_inep_avaliacao_alfabetizacao_alunos.csv"

#Chamada das Funções para a criação das Delta Tables na bronze
carregar_bronze(
    csv_path = path_meta_brasil,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_brasil"
)

carregar_bronze(
    csv_path = path_meta_municipio,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_municipio"
)

carregar_bronze(
    csv_path = path_meta_uf,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_meta_alfabetizacao_uf"
)

carregar_bronze(
    csv_path = path_avaliacao_municipio,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_municipio"
)

carregar_bronze(
    csv_path = path_avaliacao_uf,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_uf"
)

carregar_bronze(
    csv_path = path_avaliacao_alunos,
    catalogo = "workspace",
    schema = "bronze",
    tabela = "b_avaliacao_alunos"
)

In [0]:
%sql
